# Eşik Ayarlaması

👇 Çalışacağınız veri setini görmek için `player_performances.csv` oyuncu veri setini yükleyin.

In [1]:
import pandas as pd

!curl -s https://d32aokrjazspmn.cloudfront.net/materials/ML_Player_performance.csv > data/player_performances.csv

data = pd.read_csv('data/player_performances.csv')

data.head()

,games played,minutes played,points per game,field goals made,field goal attempts,field goal percent,3 point made,3 point attempt,3 point %,free throw made,free throw attempts,free throw %,offensive rebounds,defensive rebounds,rebounds,assists,steals,blocks,turnovers,target_5y
0,36,27.4,7.4,2.6,7.6,34.7,0.5,2.1,25.0,1.6,2.3,69.9,0.7,3.4,4.1,1.9,0.4,0.4,1.3,0
1,35,26.9,7.2,2.0,6.7,29.6,0.7,2.8,23.5,2.6,3.4,76.5,0.5,2.0,2.4,3.7,1.1,0.5,1.6,0
2,74,15.3,5.2,2.0,4.7,42.2,0.4,1.7,24.4,0.9,1.3,67.0,0.5,1.7,2.2,1.0,0.5,0.3,1.0,0
3,58,11.6,5.7,2.3,5.5,42.6,0.1,0.5,22.6,0.9,1.3,68.9,1.0,0.9,1.9,0.8,0.6,0.1,1.0,1
4,48,11.5,4.5,1.6,3.0,52.4,0.0,0.1,0.0,1.3,1.9,67.4,1.0,1.5,2.5,0.3,0.3,0.4,0.8,1


ℹ️ Her gözlem bir oyuncuyu temsil eder ve her sütun performansın bir özelliğidir. Hedef `target_5y`, oyuncunun 5 yıldan az [0] veya 5 yıl ve daha fazla [1] profesyonel kariyere sahip olup olmadığını tanımlar.

# Ön İşleme

👇 Ön işleme için çok fazla zaman harcamamak adına, tüm özellik setini Robust Scale ile ölçeklendirin. Bu uygulama optimal değildir, ancak ön işleme ve/veya modellerin hızla çalıştırılması için kullanılabilir.

Ölçeklendirilmiş özellik setini `X_scaled` olarak kaydedin.

In [2]:
from sklearn.preprocessing import RobustScaler

X = data.drop(columns="target_5y")
y = data["target_5y"]

X_scaled = pd.DataFrame(RobustScaler().fit_transform(X), columns=X.columns)
X_scaled.describe().round(2)

,games played,minutes played,points per game,field goals made,field goal attempts,field goal percent,3 point made,3 point attempt,3 point %,free throw made,free throw attempts,free throw %,offensive rebounds,defensive rebounds,rebounds,assists,steals,blocks,turnovers
count,1328.00,1328.00,1328.00,1328.00,1328.00,1328.00,1328.00,1328.00,1328.00,1328.00,1328.00,1328.00,1328.00,1328.00,1328.00,1328.00,1328.00,1328.00,1328.00
mean,-0.09,0.13,0.24,0.27,0.26,0.03,0.37,0.40,-0.09,0.29,0.23,-0.07,0.21,0.20,0.22,0.33,0.24,0.41,0.24
std,0.58,0.69,0.85,0.84,0.85,0.80,0.96,0.89,0.49,0.96,0.95,0.82,0.78,0.85,0.85,1.05,0.82,1.07,0.91
min,-1.73,-1.07,-0.96,-0.90,-0.95,-2.64,-0.25,-0.25,-0.69,-0.98,-1.07,-5.57,-0.80,-0.94,-0.91,-0.79,-1.00,-0.50,-1.12
25%,-0.53,-0.44,-0.37,-0.35,-0.36,-0.49,-0.25,-0.25,-0.69,-0.39,-0.43,-0.51,-0.40,-0.44,-0.41,-0.36,-0.40,-0.25,-0.38
50%,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
75%,0.47,0.56,0.63,0.65,0.64,0.51,0.75,0.75,0.31,0.61,0.57,0.49,0.60,0.56,0.59,0.64,0.60,0.75,0.62
max,0.63,2.05,4.43,4.05,3.57,3.91,5.50,5.17,2.39,6.54,6.21,2.24,4.50,4.94,4.70,6.79,4.00,9.25,4.25


### ☑️ Kodunuzu kontrol edin

In [3]:
from nbresult import ChallengeResult

result = ChallengeResult('scaled_features',
                         scaled_features = X_scaled
)

result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/lemfi/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/lemfi/projects/S16D3-S-data-threshold/tests
plugins: dash-4.3.0, typeguard-4.4.2, anyio-4.8.0
collecting ... collected 1 item

test_scaled_features.py::TestScaled_features::test_scaled_features PASSED [100%]

============================== 1 passed in 0.55s ===============================


💯 You can commit your code:

git add tests/scaled_features.pickle

git commit -m 'Completed scaled_features step'

git push origin master



# Temel Modelleme

🎯 Görev, %90 garantiyle minimum 5 yıl profesyonel olarak devam edecek oyuncuları tespit etmektir.

👇 Varsayılan bir Lojistik Regresyon modeli antrenörün gereksinimlerini karşılayacak mı? Çapraz doğrulama kullanın ve cevabınızı destekleyen skoru `base_score` değişken adı altında kaydedin.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

base_score = cross_val_score(
    LogisticRegression(max_iter=1000),
    X_scaled, y,
    cv=5,
    scoring="precision"
).mean()

print(f"base precision: {base_score:.3f}  (coach requires 0.90)")

base precision: 0.738  (coach requires 0.90)


### ☑️ Kodunuzu kontrol edin

In [5]:
from nbresult import ChallengeResult

result = ChallengeResult('base_precision',
                         score = base_score
)

result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/lemfi/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/lemfi/projects/S16D3-S-data-threshold/tests
plugins: dash-4.3.0, typeguard-4.4.2, anyio-4.8.0
collecting ... collected 1 item

test_base_precision.py::TestBase_precision::test_precision_score PASSED  [100%]

============================== 1 passed in 0.18s ===============================


💯 You can commit your code:

git add tests/base_precision.pickle

git commit -m 'Completed base_precision step'

git push origin master



# Eşik Ayarlaması

👇 Bir oyuncunun profesyonel olarak 5 yıl veya daha fazla sürmesi için %90 kesinlik garantisi veren karar eşiğini bulun. Eşiği `new_threshold` değişken adı altında kaydedin.

<details>
<summary>💡 İpucu</summary>

- [`cross_val_predict`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_predict.html) ile çapraz doğrulanmış olasılık tahminleri yapın
    
- Farklı eşiklerde kesinlik skorları oluşturmak için olasılıkları [`precision_recall_curve`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_recall_curve.html) içine yerleştirin

- 0.9 kesinliği garanti eden eşiği bulun
      
</details>

In [ ]:
# YOUR CODE HERE

### ☑️ Kodunuzu kontrol edin

In [ ]:
from nbresult import ChallengeResult

result = ChallengeResult('decision_threshold',
                         threshold = new_threshold
)

result.write()
print(result.check())

# Yeni Eşiği Kullanma

🎯 Antrenör potansiyel olarak ilginç bir oyuncu fark etti, ancak bu oyuncunun minimum 5 yıl profesyonel olarak devam edeceğine dair %90 garantinizi istiyor. Oyuncunun verilerini [buradan](https://wagon-public-datasets.s3.amazonaws.com/Machine%20Learning%20Datasets/ML_New_player.csv) indirin.

In [ ]:
new_player = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/ML_New_player.csv")

new_player

❓ Oyuncuyu antrenöre tavsiye etmeyi göze alır mısınız? Cevabınızı string olarak `recommendation` değişken adı altında "recommend" veya "not recommend" şeklinde kaydedin.

In [ ]:
# YOUR CODE HERE

### ☑️ Kodunuzu kontrol edin

In [ ]:
from nbresult import ChallengeResult

result = ChallengeResult('recommendation',
                         recommendation = recommendation
)

result.write()
print(result.check())

# 🏁